# 29 — LLM-as-Judge Eval Harness

Score a RAG pipeline on **Relevance**, **Faithfulness**, and **Completeness** using a second LLM as the judge. No labelled data required.

**What you'll learn**
- How to define a structured rubric that an LLM can apply consistently
- How `with_structured_output` forces the judge to return parseable scores
- How to aggregate per-dimension scores across a test set
- Contrast with RAGAS (example 16) which uses statistical reference-based metrics

**Companion examples:** 16-rag-eval-notebook (RAGAS), 17-corrective-rag (CRAG), 27-self-rag

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langchain-chroma chromadb python-dotenv pydantic

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os
from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get('OPENAI_API_KEY'):
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY before running'

In [ ]:
# ── 3. Knowledge base + RAG chain ─────────────────────────────────────────────
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

DOCUMENTS = [
    Document(page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs, built on top of LangChain."),
    Document(page_content="RAG (Retrieval-Augmented Generation) combines a retriever with a generative model to ground answers in source documents."),
    Document(page_content="Corrective RAG grades each retrieved document for relevance and rewrites the query if any documents score irrelevant."),
    Document(page_content="Self-RAG generates reflection tokens to decide whether retrieval is needed before fetching any documents."),
    Document(page_content="Human-in-the-loop checkpointing pauses graph execution at an interrupt node and waits for human approval before resuming."),
]

store = Chroma.from_documents(DOCUMENTS, OpenAIEmbeddings())
retriever = store.as_retriever(search_kwargs={"k": 2})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the provided context. Be concise."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

def run_rag(question: str) -> tuple[str, list]:
    docs = retriever.invoke(question)
    ctx = "\n\n".join(d.page_content for d in docs)
    answer = (rag_prompt | llm | StrOutputParser()).invoke({"context": ctx, "question": question})
    return answer, docs

print("RAG pipeline ready")

In [ ]:
# ── 4. Test dataset ───────────────────────────────────────────────────────────
TEST_SET = [
    {"question": "What is LangGraph?", "reference": "LangGraph is a library for building stateful, multi-actor applications with LLMs."},
    {"question": "How does Corrective RAG handle irrelevant documents?", "reference": "It rewrites the query when documents score irrelevant."},
    {"question": "What are reflection tokens in Self-RAG?", "reference": "Tokens that decide whether retrieval is needed before fetching documents."},
    {"question": "What happens at a human-in-the-loop interrupt?", "reference": "Graph execution pauses and waits for human approval before resuming."},
]

# Generate RAG answers for all test questions
results = []
for item in TEST_SET:
    answer, docs = run_rag(item["question"])
    results.append({**item, "answer": answer, "context": "\n".join(d.page_content for d in docs)})
    print(f"Q: {item['question']}")
    print(f"A: {answer}\n")

In [ ]:
# ── 5. Judge rubric ───────────────────────────────────────────────────────────
from pydantic import BaseModel, Field

class JudgeScore(BaseModel):
    relevance: int = Field(..., ge=0, le=3, description="0=irrelevant 1=partial 2=mostly 3=fully relevant")
    faithfulness: int = Field(..., ge=0, le=3, description="0=contradicts 1=unsupported 2=mostly 3=fully grounded")
    completeness: int = Field(..., ge=0, le=3, description="0=empty 1=partial 2=mostly 3=fully answers the question")
    reasoning: str = Field(..., description="One sentence explaining the scores")

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge = judge_llm.with_structured_output(JudgeScore)

JUDGE_PROMPT = """You are evaluating a RAG system answer. Score on three dimensions (0-3 each).

Question: {question}
Reference answer: {reference}
Retrieved context: {context}
System answer: {answer}

Relevance: does the answer address the question?
Faithfulness: is the answer grounded in the retrieved context?
Completeness: does the answer cover all key points from the reference?"""

print("Judge configured — 0-3 scale on Relevance, Faithfulness, Completeness")

In [ ]:
# ── 6. Run the judge harness ──────────────────────────────────────────────────
scored = []
for r in results:
    score = judge.invoke(JUDGE_PROMPT.format(**r))
    scored.append({**r, "score": score})
    print(f"Q: {r['question']}")
    print(f"  Relevance={score.relevance}  Faithfulness={score.faithfulness}  Completeness={score.completeness}")
    print(f"  Reasoning: {score.reasoning}\n")

In [ ]:
# ── 7. Aggregate scores ───────────────────────────────────────────────────────
n = len(scored)
avg_rel = sum(r["score"].relevance for r in scored) / n
avg_fai = sum(r["score"].faithfulness for r in scored) / n
avg_com = sum(r["score"].completeness for r in scored) / n

print(f"{'Dimension':<20} {'Avg':>6} {'Max':>6}")
print("-" * 34)
print(f"{'Relevance':<20} {avg_rel:>6.2f} {'3':>6}")
print(f"{'Faithfulness':<20} {avg_fai:>6.2f} {'3':>6}")
print(f"{'Completeness':<20} {avg_com:>6.2f} {'3':>6}")
print(f"{'Overall':<20} {(avg_rel+avg_fai+avg_com)/3:>6.2f} {'3':>6}")

## Exercises

**Exercise 1 — Add a dimension:** Add a `conciseness` score (0-3) to `JudgeScore` and update `JUDGE_PROMPT`. Re-run cells 5-7.

**Exercise 2 — Degrade the pipeline:** Change `search_kwargs={"k": 2}` to `k=0` (no retrieval). Re-run cells 3-7. Watch Faithfulness drop.

**Exercise 3 — Compare models:** Replace `"gpt-4o-mini"` in the RAG chain with a weaker model and compare scores from the judge.

## What's next

- **16-rag-eval-notebook** — RAGAS metrics (reference-based, statistical)
- **27-self-rag** — selective retrieval via reflection tokens
- **17-corrective-rag** — retrieval correction via document grading